In [6]:
from sympy import symbols, Matrix, eye, simplify
from sympy import latex
from IPython.display import display, Math

# Define symbols
dt, k_re, mass, u = symbols('dt k_re mass u_in')

A21 = symbols('A21')  # Element of matrix A
B2 = symbols('B2')  # Element of matrix B
P11, P12, P21, P22 = symbols('P11 P12 P21 P22')  # Elements of covariance matrix P
P11_sim, P12_sim, P21_sim, P22_sim = symbols('P_pred11 P_pred12 P_pred21 P_pred22')  # Simulated covariance matrix elements
Q11, Q22 = symbols('Q11 Q22')  # Elements of process noise covariance matrix Q
R_mea = symbols('R_mea')  # Measurement noise covariance (scalar)

x_pred, vx_pred = symbols('x_pred vx_pred')  # Predicted state variables
y_mea, y_meaUD = symbols('y_mea y_meaUD')  # Measurement variable
x_step, vx_step = symbols('x_step vx_step')  # relative update
# System matrices
A = Matrix([[1, dt], 
            [A21, 1]])
B = Matrix([[0], [B2]])
C = Matrix([[1, 0]])
x_pred_vector = Matrix([[x_pred], [vx_pred]])

# Prediction step (before jumping)
x_temp = A * x_pred_vector + B * u  # Predicted state vector
x_temp[0] += x_step  
P_pred = A * Matrix([[P11, P12], [P21, P22]]) * A.T + Matrix([[Q11, 0], [0, Q22]])
jumping_step = Matrix([[x_step], [vx_step]])

# Measurement step(after jumping)
# Calculate Kalman gain
S = (C * P_pred * C.T)[0, 0] + R_mea  # Extract scalar from 1x1 matrix and add R
S_sim = symbols('S')  # Measurement prediction covariance
# K = P_pred * C.T / S  # 2x1 vector

# Joseph form update
x_temp_sim_11, x_temp_sim_12 = symbols('x_temp1 x_temp2')
x_temp_sim = Matrix([[x_temp_sim_11], [x_temp_sim_12]])
K1, K2 = symbols('K1 K2')  # Kalman gain components
# K = Matrix([[P11_sim/S_sim], [P21_sim/S_sim]])
K = Matrix([[K1], [K2]])
P_pred_sim = Matrix([[P11_sim, P12_sim], [P21_sim, P22_sim]])

I = eye(2)
IKC = I - K * C
P_update = IKC * P_pred_sim * IKC.T + K * R_mea * K.T

# compute update state vector
x_update = x_temp_sim + K * (y_meaUD - (C * (x_temp_sim))[0, 0])
x_update[1] += jumping_step[1]

# Simplify expressions
# Simplify all key expressions
x_temp_s = simplify(x_temp)
P_pred_s = simplify(P_pred)
S_s = simplify(S)
K_s = simplify(K)
P_update_s = simplify(P_update)
x_update_s = simplify(x_update)

# Display results in LaTeX
display(Math(r"x_{\text{temp}}: " + latex(x_temp_s)))
display(Math(r"P_{\text{pred}}: " + latex(P_pred_s)))
display(Math(r"S: " + latex(S_s)))
display(Math(r"K: " + latex(K_s)))
display(Math(r"P_{\text{update}}: " + latex(P_update_s)))
display(Math(r"x_{\text{update}}: " + latex(x_update_s)))
print("x_temp:", x_temp_s)
print("P_pred:", P_pred_s)
print("S:", S_s)
print("K:", K_s)
print("P_update:", P_update_s)
print("x_update:", x_update_s)






<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

x_temp: Matrix([[dt*vx_pred + x_pred + x_step], [A21*x_pred + B2*u_in + vx_pred]])
P_pred: Matrix([[P11 + P21*dt + Q11 + dt*(P12 + P22*dt), A21*(P11 + P21*dt) + P12 + P22*dt], [A21*P11 + P21 + dt*(A21*P12 + P22), A21*P12 + A21*(A21*P11 + P21) + P22 + Q22]])
S: P11 + P21*dt + Q11 + R_mea + dt*(P12 + P22*dt)
K: Matrix([[K1], [K2]])
P_update: Matrix([[K1**2*R_mea + P_pred11*(K1 - 1)**2, K1*K2*R_mea + K2*P_pred11*(K1 - 1) - P_pred12*(K1 - 1)], [K1*K2*R_mea + (K1 - 1)*(K2*P_pred11 - P_pred21), K2**2*R_mea - K2*P_pred12 + K2*(K2*P_pred11 - P_pred21) + P_pred22]])
x_update: Matrix([[-K1*(x_temp1 - y_meaUD) + x_temp1], [-K2*(x_temp1 - y_meaUD) + vx_step + x_temp2]])


In [2]:
import numpy as np

def sqrt_diagonal_matrix(D):
    # Check if D is a square matrix
    assert D.shape[0] == D.shape[1], "Matrix must be square"
    # Extract diagonal elements
    diag_elements = np.diag(D)
    # Compute square roots of diagonal elements
    sqrt_diag = np.sqrt(diag_elements)
    # Construct the square root diagonal matrix
    sqrt_D = np.diag(sqrt_diag)
    return sqrt_D

# Example usage
D = np.array([[4, 0], [0, 9]])
sqrt_D = sqrt_diagonal_matrix(D)
print(sqrt_D)


[[2. 0.]
 [0. 3.]]


In [1]:
from sympy import symbols, Matrix, cse, numbered_symbols
from sympy.utilities.codegen import codegen
from sympy.printing import ccode

def generate_kalman_filter_code():
    # Define symbolic variables
    x0, x1 = symbols('x0 x1')  # State variables
    P00, P01, P10, P11 = symbols('P00 P01 P10 P11')  # Covariance matrix
    H00, H01, H10, H11 = symbols('H00 H01 H10 H11')  # Measurement matrix
    R00, R01, R10, R11 = symbols('R00 R01 R10 R11')  # Measurement noise
    z0, z1 = symbols('z0 z1')  # Measurement
    
    # Create matrices
    x = Matrix([x0, x1])
    P = Matrix([[P00, P01], [P10, P11]])
    H = Matrix([[H00, H01], [H10, H11]])
    R = Matrix([[R00, R01], [R10, R11]])
    z = Matrix([z0, z1])
    
    # Kalman filter measurement update equations
    # Innovation covariance
    S = H * P * H.T + R
    
    # Efficient 2x2 inversion
    det_S = S[0,0] * S[1,1] - S[0,1] * S[1,0]
    invS = Matrix([
        [ S[1,1]/det_S, -S[0,1]/det_S],
        [-S[1,0]/det_S,  S[0,0]/det_S]
    ])
    
    # Kalman gain
    K = P * H.T * invS
    
    # State update
    y = z - H * x
    x_new = x + K * y
    
    # Covariance update
    I = Matrix([[1, 0], [0, 1]])  # Identity matrix
    P_new = (I - K * H) * P
    
    # Extract components for code generation
    x_new0 = x_new[0]
    x_new1 = x_new[1]
    P_new00 = P_new[0,0]
    P_new01 = P_new[0,1]
    P_new10 = P_new[1,0]
    P_new11 = P_new[1,1]
    
    # Use Common Subexpression Elimination (CSE) to optimize the code
    expressions = [x_new0, x_new1, P_new00, P_new01, P_new10, P_new11]
    replacements, reduced_exprs = cse(expressions, numbered_symbols('t'))
    
    # Generate C code
    [(c_name, c_code), (h_name, c_header)] = codegen(
        [('x_new0', reduced_exprs[0]),
         ('x_new1', reduced_exprs[1]),
         ('P_new00', reduced_exprs[2]),
         ('P_new01', reduced_exprs[3]),
         ('P_new10', reduced_exprs[4]),
         ('P_new11', reduced_exprs[5])],
        language='C',
        prefix='kalman_filter',
        project='KalmanFilter',
        to_files=False
    )
    
    # Add intermediate variables to the C code
    full_c_code = "void kalman_update(\n"
    full_c_code += "    double x0, double x1, \n"
    full_c_code += "    double P00, double P01, double P10, double P11, \n"
    full_c_code += "    double H00, double H01, double H10, double H11, \n"
    full_c_code += "    double R00, double R01, double R10, double R11, \n"
    full_c_code += "    double z0, double z1, \n"
    full_c_code += "    double* x_new0, double* x_new1, \n"
    full_c_code += "    double* P_new00, double* P_new01, double* P_new10, double* P_new11) {\n\n"
    
    # Add intermediate variable declarations
    for var, expr in replacements:
        full_c_code += f"    double {ccode(var)} = {ccode(expr)};\n"
    
    # Add the main expressions
    full_c_code += f"\n    *x_new0 = {ccode(reduced_exprs[0])};\n"
    full_c_code += f"    *x_new1 = {ccode(reduced_exprs[1])};\n"
    full_c_code += f"    *P_new00 = {ccode(reduced_exprs[2])};\n"
    full_c_code += f"    *P_new01 = {ccode(reduced_exprs[3])};\n"
    full_c_code += f"    *P_new10 = {ccode(reduced_exprs[4])};\n"
    full_c_code += f"    *P_new11 = {ccode(reduced_exprs[5])};\n"
    full_c_code += "}\n"
    
    return full_c_code

# Generate and print the C code
c_code = generate_kalman_filter_code()
print("Generated C code:")
print(c_code)

# Optional: Save to file
with open("kalman_filter.c", "w") as f:
    f.write(c_code)

Generated C code:
void kalman_update(
    double x0, double x1, 
    double P00, double P01, double P10, double P11, 
    double H00, double H01, double H10, double H11, 
    double R00, double R01, double R10, double R11, 
    double z0, double z1, 
    double* x_new0, double* x_new1, 
    double* P_new00, double* P_new01, double* P_new10, double* P_new11) {

    double t0 = -H10*x0 - H11*x1 + z1;
    double t1 = H00*P00;
    double t2 = H01*P10 + t1;
    double t3 = H01*P11;
    double t4 = H00*P01 + t3;
    double t5 = H00*t2 + H01*t4 + R00;
    double t6 = H10*P00;
    double t7 = H11*P10 + t6;
    double t8 = H11*P11;
    double t9 = H10*P01 + t8;
    double t10 = H10*t7 + H11*t9 + R11;
    double t11 = H00*t7;
    double t12 = H01*t9;
    double t13 = H10*t2;
    double t14 = H11*t4;
    double t15 = 1.0/(t10*t5 - (R01 + t13 + t14)*(R10 + t11 + t12));
    double t16 = t15*(H11*P01 + t6);
    double t17 = -R01 - t13 - t14;
    double t18 = t15*(H01*P01 + t1);
    double t19 = t16*